# BW \#105 Federal employees
Elon Musk has been tasked with cutting the size of government, and he has started to do it, despite federal judges saying that he and his team are breaking multiple laws, and need to justify their actions in court before continuing.

## Data and six questions
This week, we'll thus look at data about the federal workforce, using data from the Office of Personnel Management (OPM), which is in charge of federal employees. We'll look at where they live, what sorts of work they do, and how much money they earn.

## Challenges
The learning goals include working with multiple files, grouping, joins, plotting, and string manipulation.

- Create a data frame describing all federal employees from the main `FACTDATA_MAR2024.TXT` file. For each of the columns 'LOC', 'AGELVL', 'EDLVL', 'LOSLVL', 'OCC', 'SALLVL', and 'STEMOCC', load the corresponding data file (i.e., DTname.txt) into a data frame, and combine it with the main data frame. Do the same for DTagy.txt, which should be combined using the AGYSUB column in the main file.

- Find the 20 top level agencies with the greates number of employees. Print their names ("agency translation") and the number of people who work there, with commas before every three digits. 

### Create a data frame describing all federal employees from the main `FACTDATA_MAR2024.TXT file`. For each of the columns 'LOC', 'AGELVL', 'EDLVL', 'LOSLVL', 'OCC', 'SALLVL', and 'STEMOCC', load the corresponding data file (i.e., DTname.txt) into a data frame, and combine it with the main data frame. Do the same for DTagy.txt, which should be combined using the AGYSUB column in the main file.

to read from a bunch of files in the same directory use `os.path.join` to get a full pathname from a combination of directory and filename.

use PyArrow for loading the csv into a df bc it's much faster than using the native pandas csv loader. Plus it automatically identifies datetime columns

`pd.read_csv` can read zipfile containing a single csv file by automatically expand and read it. 

The `\opm-data\DTloc.txt` file contains 4 columns LOCTYP,LOCTYPT,LOC,LOCT

#### How to merge data from 8 others ? 
- `join` : combines them much as is done in SQL. Use the index of both dfs
- `merge` : does the same but can work on any two columns not just the indexes

In [1]:
import pandas as pd
import os 

dirname = './opm-data'
filename = 'FACTDATA_MAR2024.TXT'
df = pd.read_csv(os.path.join(dirname, filename), engine = 'pyarrow')
df.head()

,AGYSUB,LOC,AGELVL,EDLVL,GSEGRD,LOSLVL,OCC,PATCO,PP,PPGRD,SALLVL,STEMOCC,SUPERVIS,TOA,WORKSCH,WORKSTAT,DATECODE,EMPLOYMENT,SALARY,LOS
0,AA00,11,F,13,None,G,0340,2,ES,ES-**,20,XXXX,2,50,F,1,202403,1,210000.0,20.8
1,AA00,11,J,15,None,I,0905,1,ES,ES-**,20,XXXX,2,50,F,1,202403,1,203000.0,31.2
2,AA00,11,K,04,None,G,0301,2,99,EX-02,30,XXXX,2,48,F,2,202403,1,NaN,22.0
3,AA00,11,C,04,12,B,0560,2,99,GS-12,9,XXXX,8,15,F,1,202403,1,99200.0,2.5
4,AA00,11,D,04,13,B,0905,1,99,GS-13,11,XXXX,8,30,F,1,202403,1,117962.0,2.5


In [2]:
join_columns = ['LOC', 'AGELVL', 'EDLVL', 'LOSLVL', 'OCC', 'SALLVL', 'STEMOCC']

for one_col in join_columns:
    print(one_col)
    small_filename = os.path.join(dirname, f'DT{one_col.lower()}.txt')
    small_df = pd.read_csv(small_filename, engine='pyarrow')
    df = df.merge(small_df, on = one_col)

LOC
AGELVL
EDLVL
LOSLVL
OCC
SALLVL
STEMOCC


In [3]:
agy_df = pd.read_csv(os.path.join(dirname, 'DTagy.txt'), engine='pyarrow')
df = df.merge(agy_df, on = 'AGYSUB')

In [6]:
df.shape

(2278730, 44)

In [7]:
df.head()

,AGYSUB,LOC,AGELVL,EDLVL,GSEGRD,LOSLVL,OCC,PATCO,PP,PPGRD,...,STEMAGG,STEMAGGT,STEMTYP,STEMTYPT,STEMOCCT,AGYTYP,AGYTYPT,AGY,AGYT,AGYSUBT
0,AA00,11,F,13,None,G,0340,2,ES,ES-**,...,3,ALL OTHER OCCUPATIONS,6,ALL OTHER OCCUPATIONS,XXXX-ALL OTHER OCCUPATIONS,4,Small Independent Agencies (less than 100 empl...,AA,AA-ADMINISTRATIVE CONFERENCE OF THE UNITED STATES,AA00-ADMINISTRATIVE CONFERENCE OF THE UNITED S...
1,AA00,11,J,15,None,I,0905,1,ES,ES-**,...,3,ALL OTHER OCCUPATIONS,6,ALL OTHER OCCUPATIONS,XXXX-ALL OTHER OCCUPATIONS,4,Small Independent Agencies (less than 100 empl...,AA,AA-ADMINISTRATIVE CONFERENCE OF THE UNITED STATES,AA00-ADMINISTRATIVE CONFERENCE OF THE UNITED S...
2,AA00,11,K,04,None,G,0301,2,99,EX-02,...,3,ALL OTHER OCCUPATIONS,6,ALL OTHER OCCUPATIONS,XXXX-ALL OTHER OCCUPATIONS,4,Small Independent Agencies (less than 100 empl...,AA,AA-ADMINISTRATIVE CONFERENCE OF THE UNITED STATES,AA00-ADMINISTRATIVE CONFERENCE OF THE UNITED S...
3,AA00,11,C,04,12,B,0560,2,99,GS-12,...,3,ALL OTHER OCCUPATIONS,6,ALL OTHER OCCUPATIONS,XXXX-ALL OTHER OCCUPATIONS,4,Small Independent Agencies (less than 100 empl...,AA,AA-ADMINISTRATIVE CONFERENCE OF THE UNITED STATES,AA00-ADMINISTRATIVE CONFERENCE OF THE UNITED S...
4,AA00,11,D,04,13,B,0905,1,99,GS-13,...,3,ALL OTHER OCCUPATIONS,6,ALL OTHER OCCUPATIONS,XXXX-ALL OTHER OCCUPATIONS,4,Small Independent Agencies (less than 100 empl...,AA,AA-ADMINISTRATIVE CONFERENCE OF THE UNITED STATES,AA00-ADMINISTRATIVE CONFERENCE OF THE UNITED S...


After all this, the data frame (df) contained 2,278,730 rows and 44 columns.

### Find the 20 top-level agencies with the greatest number of employees. Print their names ("agency translation") and the number of people who work there, with commas before every three digits.

In [9]:
(df['AGYT'].value_counts().head(20).apply(lambda x:f'{x:,}'))

AGYT
VA-DEPARTMENT OF VETERANS AFFAIRS                   486,522
HS-DEPARTMENT OF HOMELAND SECURITY                  222,539
AR-DEPARTMENT OF THE ARMY                           221,037
NV-DEPARTMENT OF THE NAVY                           216,537
AF-DEPARTMENT OF THE AIR FORCE                      168,505
DD-DEPARTMENT OF DEFENSE                            156,803
DJ-DEPARTMENT OF JUSTICE                            116,614
TR-DEPARTMENT OF THE TREASURY                       108,869
AG-DEPARTMENT OF AGRICULTURE                         92,072
HE-DEPARTMENT OF HEALTH AND HUMAN SERVICES           91,058
IN-DEPARTMENT OF THE INTERIOR                        62,890
SZ-SOCIAL SECURITY ADMINISTRATION                    59,227
TD-DEPARTMENT OF TRANSPORTATION                      55,806
CM-DEPARTMENT OF COMMERCE                            47,650
NN-NATIONAL AERONAUTICS AND SPACE ADMINISTRATION     18,073
DN-DEPARTMENT OF ENERGY                              16,846
EP-ENVIRONMENTAL PROTECTION AGENCY 